In [1]:
import math
import pyfastx
from Bio import Align
from Bio.Seq import Seq
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.backends.backend_pdf import PdfPages

### Overview
These functions take as input a fastq file containing reads mapping to a given locus, and a fasta file with expected/validated sequences for that locus. The reads and expected sequences (and the associated base qualities) need to be in the same orientation and be trimmed so they cover the same genomic region (this is done for HYP1 by parse_hyp_motifs.py).

Then, we align each fastq read against every sequence in the fasta file and choose whichever fasta sequence gives the best alignment (as the most likely ground truth). Then we write the read-vs-fasta-sequence alignment into a PDF file and colour the read by base quality. Repeating this for every read in the fastq file produces a large PDF that can be used for manual examination, that is, checking whether the differences between a read and the best-aligning fasta sequence are embedded in low base qualities (suggesting sequencing errors) or whether they are embedded in high base qualities (suggesting biological variation).

Note that the amino acid sequence written across the top of the alignment may not work for non-HYP genes, because to make sure it is in frame we check for the presence of the amino acid sequence GGG (as in YERGGG, YEHGGG, SDRGGG)

In [2]:
def get_best_alignment(fastq_seq, fasta_records, aligner):
    """
    Performs global alignment against all FASTA sequences and returns the best one.
    """
    best_score = -float('inf')
    best_aln = None
    best_fasta_name = ""
    
    for fasta_name, fasta_seq in fasta_records:
        # Perform Needleman-Wunsch global alignment
        aln = aligner.align(fasta_seq, fastq_seq)[0]
        if aln.score > best_score:
            best_score = aln.score
            best_aln = aln
            best_fasta_name = fasta_name

    # best_aln[0] is the target (FASTA), best_aln[1] is the query (FASTQ)
    return str(best_aln[0]), str(best_aln[1]), best_fasta_name

def process_qualities(aligned_fastq, fastq_qual_string):
    """
    Maps the PHRED quality scores to the gapped FASTQ alignment.
    Assumes standard Phred+33 encoding.
    """
    qual_scores = [ord(c) - 33 for c in fastq_qual_string]
    qual_list = []
    q_idx = 0
    
    for char in aligned_fastq:
        if char == '-':
            qual_list.append(None) # Gaps have no quality score
        else:
            qual_list.append(qual_scores[q_idx])
            q_idx += 1
            
    return qual_list

def trim_alignment(aligned_fasta, aligned_fastq, qual_list, trim_start=12, trim_end=13):
    """
    Trims `trim_start` bases from the beginning and `trim_end` bases from the end.
    """
    if len(aligned_fasta) > (trim_start + trim_end):
        # Handle the case where trim_end is 0 to avoid slicing :-0 (which yields an empty string)
        end_idx = -trim_end if trim_end > 0 else None
        return (
            aligned_fasta[trim_start:end_idx], \
            aligned_fastq[trim_start:end_idx], \
            qual_list[trim_start:end_idx]
        )
    return aligned_fasta, aligned_fastq, qual_list

def create_protein_string(aln_fasta):
    """
    Translates the FASTA sequence, dynamically finding the correct frame 
    by checking for the presence of 'GGG'.
    """
    raw_fasta = aln_fasta.replace("-", "")
    
    best_frame = 0
    best_protein_seq = ""
    
    # Check frames 0, 1, and 2
    for frame in range(3):
        seq_to_translate = raw_fasta[frame:]
        
        # Pad to a multiple of 3 to avoid translation warnings
        remainder = len(seq_to_translate) % 3
        if remainder != 0:
            seq_to_translate += "N" * (3 - remainder)
            
        protein_seq = str(Seq(seq_to_translate).translate())
        
        # Check condition or default to frame 2 if loop exhausts
        if "GGG" in protein_seq or frame == 2:
            best_frame = frame
            best_protein_seq = protein_seq
            if "GGG" in protein_seq:
                break
                
    # Map the valid frame back onto the gapped sequence string
    aa_list = []
    nt_count = 0
    for char in aln_fasta:
        if char == '-':
            aa_list.append(' ')
        else:
            if nt_count < best_frame:
                aa_list.append(' ') # Skip bases before the determined frame
            else:
                adjusted_nt_count = nt_count - best_frame
                if adjusted_nt_count % 3 == 1:  # Place over the middle nucleotide of the codon
                    aa_idx = adjusted_nt_count // 3
                    aa = best_protein_seq[aa_idx] if aa_idx < len(best_protein_seq) else ' '
                    aa_list.append(aa)
                else:
                    aa_list.append(' ')
            nt_count += 1
            
    return "".join(aa_list)

def create_middle_match_string(aligned_fasta, aligned_fastq):
    """
    Generates the middle standard comparison string (EMBOSS style).
    '|' for match, ':' for mismatch, '-' for indels.
    """
    middle = []
    for a, b in zip(aligned_fasta, aligned_fastq):
        if a == b and a != '-':
            middle.append('|')
        elif a == '-' or b == '-':
            middle.append('-')
        else:
            middle.append(':')
    return "".join(middle)

def plot_single_alignment(pdf, aln_fasta, aln_fastq, qual_list, aln_protein, fq_name, fa_name):
    """
    Plots a wrapped, coloured alignment to the active PDF.
    Plot width is strictly forced to exactly 180 mm. Length scales dynamically.
    """
    # Force EXACTLY 18 cm width (18.0 / 2.54 inches)
    fig_width_inch = 18.0 / 2.54 
    chars_per_line = 85
    num_chunks = math.ceil(len(aln_fasta) / chars_per_line)
    
    # Precise physical margins and sizing in inches to close layout gaps
    top_margin_inch = 0.5
    bottom_margin_inch = 0.7  # Allocation for colorbar + legend text room
    chunk_height_inch = 0.6
    ax_height_inch = num_chunks * chunk_height_inch
    
    fig_height = top_margin_inch + bottom_margin_inch + ax_height_inch
    
    fig, ax = plt.subplots(figsize=(fig_width_inch, fig_height))
    middle_str = create_middle_match_string(aln_fasta, aln_fastq)
    
    # Define Custom Colormap and linear normalization
    cmap = mcolors.LinearSegmentedColormap.from_list("phred_cmap", ["red", "yellow", "cyan", "cornflowerblue"])
    norm = mcolors.Normalize(vmin=1, vmax=40)
    
    y_offset = 0
    font_kwargs = dict(family='monospace', size=8, ha='center', va='center')
    
    for chunk in range(num_chunks):
        start = chunk * chars_per_line
        end = min(start + chars_per_line, len(aln_fasta))
        
        # Plot each character iteratively to perfectly align them in a grid
        for i, idx in enumerate(range(start, end)):
            # 0. Protein Sequence (Topmost)
            ax.text(i, y_offset + 1, aln_protein[idx], color='purple', **font_kwargs)
            
            # 1. FASTA Sequence
            ax.text(i, y_offset, aln_fasta[idx], **font_kwargs)
            
            # 2. Match Symbols (Middle)
            mid_char = middle_str[idx]
            mid_color = 'red' if mid_char in [':', '-'] else 'black'
            ax.text(i, y_offset - 1, mid_char, color=mid_color, **font_kwargs)
            
            # 3. FASTQ Sequence (Bottom, quality-colored background)
            c = aln_fastq[idx]
            q = qual_list[idx]
            
            if c == '-' or q is None:
                bg_color = 'white'
            else:
                bg_color = cmap(norm(q))
                
            # pad=0.0 narrows the background box to perfectly fit the character width 
            ax.text(i, y_offset - 2, c, color='black', 
                    bbox=dict(boxstyle="square,pad=0.0", facecolor=bg_color, edgecolor='none'),
                    **font_kwargs)
            
        y_offset -= 5 # Drop down for the next wrapped line
        
    # Configure axes limits tightly around text elements to eliminate white space
    ax.set_xlim(-1, chars_per_line)
    ax.set_ylim(y_offset + 2.5, 1.5)
    ax.axis('off')
    
    # Title
    ax.set_title(f"Query variant allele: {fq_name} \nBest-aligning germline allele: {fa_name}", 
                 fontsize=9, weight='bold', loc='left')

    # Add Colorbar Legend at the bottom
    # Placed safely inside the bottom margin to guarantee labels never get clipped
    cax_bottom = 0.4 / fig_height 
    cax_height = 0.08 / fig_height 
    cax = fig.add_axes([0.15, cax_bottom, 0.7, cax_height])
    
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
    cbar.set_label('Phred-scaled base quality score', fontsize=8)
    cbar.ax.tick_params(labelsize=8)
    
    # Adjust layout boundaries exactly based on physical inch configurations
    plt.subplots_adjust(
        left=0.05, 
        right=0.95, 
        top=1.0 - (top_margin_inch / fig_height), 
        bottom=bottom_margin_inch / fig_height
    )
    
    # Save to PDF (Omitting bbox_inches='tight' locks width directly to 18cm)
    pdf.savefig(fig, dpi=300)
    plt.close(fig)

def generate_alignment_report(fasta_path, fastq_path, output_pdf_path, trim_start=11, trim_end=11):
    """
    Top-level wrapper function to generate a multi-page PDF with read vs ground truth alignments for a fastq file and a ground truth fasta
    """
    # 1. Parse FASTA Sequences and close handle immediately
    fasta_obj = pyfastx.Fasta(fasta_path, build_index=False)
    fasta_records = [(name, seq) for name, seq in fasta_obj]
    del fasta_obj 
    
    # 2. Configure the Global Aligner
    aligner = Align.PairwiseAligner()
    aligner.mode = 'global'
    aligner.match_score = 2
    aligner.mismatch_score = -1
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -1
    
    # 3. Process FASTQ and Plot
    fastq_records = pyfastx.Fastq(fastq_path, build_index=False)

    mapq_data = []
    
    try:
        with PdfPages(output_pdf_path) as pdf:
            for fq_name, fq_seq, fq_qual in fastq_records:
                # Step A: Align
                aln_fasta, aln_fastq, best_fa_name = get_best_alignment(
                    fq_seq, fasta_records, aligner
                )
                
   # Step C: Map PHRED scores to gapped alignment
                qual_list = process_qualities(aln_fastq, fq_qual)
                
                # Step D: Trim alignment first
                aln_fasta, aln_fastq, qual_list = trim_alignment(
                    aln_fasta, aln_fastq, qual_list, trim_start=trim_start, trim_end=trim_end
                )

                # Step E: Generate Translated Protein String AFTER trimming
                aln_protein = create_protein_string(aln_fasta)
                
                # Step F: Plot and append to PDF
                plot_single_alignment(pdf, aln_fasta, aln_fastq, qual_list, aln_protein, fq_name, best_fa_name)
    finally:
        del fastq_records
        
    return

In [ ]:
generate_alignment_report('pallida.top6_alleles.fasta', 'pallida.example_alleles.fastq', 'output.pdf')